In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# -------------------------
# Load TEA-seq Labels
# -------------------------
mdata = pd.read_csv("../data/cleaned_cell_labels_meta_tea_seq.csv", index_col=0)
y = mdata.to_numpy().flatten()
n = len(y)
idx_all = np.arange(n)

In [ ]:
# make sure splits directory exists
Path("../splits").mkdir(exist_ok=True, parents=True)

In [ ]:
# -------------------------
# Define split specifications
# -------------------------
split_specs = {
    "split3_all_celltypes": {"allowed_celltypes": None},
}

# -------------------------
# Helper to create and save splits excluding Hyper-indices
# -------------------------

def make_and_save_split_exclude_hyper(split_name, allowed_celltypes):
    # 1. Get the indices for the specific subset
    if allowed_celltypes is None:
        mask_subset = np.ones_like(y, dtype=bool)
    else:
        mask_subset = np.isin(y, list(allowed_celltypes))
    
    idx_subset = idx_all[mask_subset]

    # 2. Load hyperparameter indices to exclude
    hyper_path = f"../splits/tea_{split_name}_hyper_idx.csv"
    if not Path(hyper_path).exists():
        print(f"Warning: {hyper_path} not found. Skipping {split_name}.")
        return

    idx_hyper = pd.read_csv(hyper_path)["index"].values

    # 3. Remove hyper-indices from the subset
    # np.setdiff1d returns elements in idx_subset that are NOT in idx_hyper
    idx_remaining = np.setdiff1d(idx_subset, idx_hyper)
    y_remaining = y[idx_remaining]

    print(f"{split_name}: {idx_subset.size} total, {len(idx_hyper)} excluded (hyper), {len(idx_remaining)} remaining for train/test.")

    # 4. Perform Train-Test Split (80/20) on the remaining cells
    idx_tr_sub, idx_te_sub = train_test_split(
        idx_remaining,
        test_size=0.2,
        random_state=42,
        stratify=y_remaining
    )

    # 5. Save the CSVs
    pd.DataFrame({"index": idx_tr_sub}).to_csv(
        f"../splits/tea_{split_name}_train_idx.csv", index=False
    )
    pd.DataFrame({"index": idx_te_sub}).to_csv(
        f"../splits/tea_{split_name}_test_idx.csv", index=False
    )

    print(f"  Saved train ({len(idx_tr_sub)}) and test ({len(idx_te_sub)}) indices for {split_name}.")

# -------------------------
# Run
# -------------------------
for split_name, conf in split_specs.items():
    make_and_save_split_exclude_hyper(split_name, conf["allowed_celltypes"])